F1 Shared state & config
Phase 1
What: one AgentState object that flows through every node, plus a config that reads keys from .env.

Why: every agent reads and updates the same state (question, documents, results, steps, revision count). No shared state = no collaboration.

In [1]:
!pip install -q langgraph langchain langchain-google-genai python-dotenv qdrant-client pydantic


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from typing import TypedDict, List, Optional
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()  # .env dan kalitlarni o'qiydi

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "GOOGLE_API_KEY topilmadi — .env faylni tekshir!"

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", google_api_key=GOOGLE_API_KEY)

class AgentState(TypedDict):
    question: str
    plan: str
    documents: List[str]
    sql_result: Optional[str]
    code_result: Optional[str]
    answer: Optional[str]
    steps: List[str]
    revisions: int

print("Setup muvaffaqiyatli! LLM tayyor.")

Setup muvaffaqiyatli! LLM tayyor.


In [3]:
response = llm.invoke("Salom! Bir gapda o'zingni tanit.")
print(response.content)

[{'type': 'text', 'text': "Salom, men sun'iy intellekt asosida ishlovchi raqamli yordamchingizman va sizga turli savollarda yordam berish, ma'lumot izlash hamda ijodiy vazifalarni bajarish uchun xizmat qilaman.", 'extras': {'signature': 'EjQKMgERTTIPbaJCNY15VZtg2upIRhKcRM/g9eHxt4qTdN1POVo/j1UKc0cKHXT8xl25SRWH'}}]


F2 Ingestion & vector store
Phase 1
What: load documents → chunk → embed → store in Qdrant.

Why: the retriever agent has nothing to search until documents are embedded and stored.

In [4]:
import os

os.makedirs("data/docs", exist_ok=True)

docs_content = {
    "refund_policy.txt": """TechNova Qaytarish Siyosati
Mijozlar mahsulotni sotib olgandan keyin 30 kun ichida qaytarishi mumkin.
Qaytarish uchun mahsulot original qadoqda bo'lishi kerak.
Pul qaytarish 5-7 ish kuni ichida amalga oshiriladi.
Elektron mahsulotlar (noutbuklar, telefonlar) 14 kun ichida qaytariladi, agar zarar ko'rmagan bo'lsa.""",

    "product_faq.txt": """TechNova Mahsulotlar bo'yicha savol-javob
TechNova 2020-yilda tashkil topgan, dasturiy ta'minot va IT-uskunalar sotadi.
Asosiy mahsulotlar: bulutli xotira xizmati (CloudSafe), CRM tizimi (SalesPro), noutbuklar.
CloudSafe narxi: oyiga $9.99 dan boshlanadi, 1TB xotira bilan.
SalesPro CRM kichik va o'rta biznes uchun mo'ljallangan, oyiga $29 dan.
Texnik yordam 24/7 chat orqali, yoki support@technova.com email orqali.""",

    "churn_report.txt": """TechNova Mijozlar Yo'qotish (Churn) Tahlili — 2025 4-chorak
2025-yil 4-chorakda mijozlar yo'qotish darajasi 8.2% ni tashkil etdi, bu oldingi chorakdan yuqori.
Asosiy sabablar: (1) narxlar raqobatchilarga nisbatan yuqori deb topilgan, (2) mijozlar
texnik yordam javob berish tezligidan norozi bo'lgan, (3) SalesPro'da kerakli integratsiyalar yo'qligi.
Eng ko'p yo'qotish kichik biznes segmentida kuzatilgan — ular narxga sezgir.
Yirik korporativ mijozlar orasida yo'qotish past, atigi 2.1%."""
}

for filename, content in docs_content.items():
    with open(f"data/docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("3 ta hujjat yaratildi:", os.listdir("data/docs"))

3 ta hujjat yaratildi: ['churn_report.txt', 'product_faq.txt', 'refund_policy.txt']


In [5]:
!pip install -q langchain-community langchain-text-splitters


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# 1. Hujjatlarni yuklash
loader = DirectoryLoader("data/docs", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
docs = loader.load()
print(f"{len(docs)} ta hujjat yuklandi")

# 2. Chunk qilish
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} ta chunk yaratildi")

# 3. Embeddings modeli (Gemini)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
    output_dimensionality=768
)

# 4. Qdrant vector store (embedded, signup shart emas)
client = QdrantClient(location=":memory:")
client.create_collection(
    collection_name="technova_docs",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="technova_docs",
    embedding=embeddings
)
vectorstore.add_documents(chunks)
print("Qdrant vector store tayyor!")

C:\Users\user\AppData\Local\Temp\ipykernel_1496\267865251.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


3 ta hujjat yuklandi
3 ta chunk yaratildi
Qdrant vector store tayyor!


In [7]:
!pip install -q langchain-qdrant


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Similarity search test
query = "Mahsulotni qanday qaytarish mumkin?"
results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results):
    print(f"--- Natija {i+1} ---")
    print(doc.page_content[:200])
    print()

--- Natija 1 ---
TechNova Qaytarish Siyosati
Mijozlar mahsulotni sotib olgandan keyin 30 kun ichida qaytarishi mumkin.
Qaytarish uchun mahsulot original qadoqda bo'lishi kerak.
Pul qaytarish 5-7 ish kuni ichida amalga

--- Natija 2 ---
TechNova Mahsulotlar bo'yicha savol-javob
TechNova 2020-yilda tashkil topgan, dasturiy ta'minot va IT-uskunalar sotadi.
Asosiy mahsulotlar: bulutli xotira xizmati (CloudSafe), CRM tizimi (SalesPro), n



F3 Retriever agent
Phase 2
What: the RAG agent — retrieves the top-k relevant chunks for the question.

Why: it answers "from your documents" questions; the reused core from the previous project.

In [9]:
def retriever_agent(state: AgentState):
    docs = vectorstore.similarity_search(state["question"], k=4)
    return {
        "documents": [d.page_content for d in docs],
        "steps": state["steps"] + ["retriever"]
    }

# Yakka holda test qilish
test_state: AgentState = {
    "question": "Mahsulotni qanday qaytarish mumkin?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}

result = retriever_agent(test_state)
print("Topilgan hujjatlar soni:", len(result["documents"]))
print("Steps:", result["steps"])
print("\nBirinchi hujjat:")
print(result["documents"][0][:200])

Topilgan hujjatlar soni: 3
Steps: ['retriever']

Birinchi hujjat:
TechNova Qaytarish Siyosati
Mijozlar mahsulotni sotib olgandan keyin 30 kun ichida qaytarishi mumkin.
Qaytarish uchun mahsulot original qadoqda bo'lishi kerak.
Pul qaytarish 5-7 ish kuni ichida amalga


F4 Web agent
Phase 2
What: a Tavily web-search agent for questions outside the documents.

Why: extends coverage beyond the local knowledge base.

In [10]:
!pip install -q tavily-python


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
import os
from tavily import TavilyClient

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

def web_agent(state: AgentState):
    if not TAVILY_API_KEY:
        print("Tavily key topilmadi, web qidiruv o'tkazib yuborildi.")
        return {"steps": state["steps"] + ["web(skipped)"]}
    
    client = TavilyClient(api_key=TAVILY_API_KEY)
    hits = client.search(state["question"] + " 2026 eng so'nggi natija", search_depth="advanced")["results"]
    return {
        "documents": state["documents"] + [h["content"] for h in hits],
        "steps": state["steps"] + ["web"]
    }

# Yakka holda test qilish
test_state: AgentState = {
    "question": "2026-yilda sun'iy intellekt bozori qanday rivojlanmoqda?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}

result = web_agent(test_state)
print("Steps:", result["steps"])
print("Topilgan natijalar soni:", len(result.get("documents", [])))
if result.get("documents"):
    print("\nBirinchi natija:")
    print(result["documents"][0][:200])

Steps: ['web']
Topilgan natijalar soni: 5

Birinchi natija:
Ulashish:

#Sunʼiy intelekt

## Mavzuga oid materiallar

Chilida sun’iy intellekt sug‘orish paytida suv sarfini 54% ga kamaytirdi

Voqea-hodisalar

Chilida sun’iy intellekt sug‘orish paytida suv sarfi


In [12]:
from dotenv import load_dotenv
load_dotenv(override=True)

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print("Tavily key topildimi:", bool(TAVILY_API_KEY))

Tavily key topildimi: True


F5 Data agent — Text-to-SQL
Phase 2
What: the hard new skill — the agent writes a SQL query from the question, runs it on a real database, reads the result.

Why: numeric/aggregate questions ("how many…", "average…") can only be answered by querying data, not by RAG.

In [13]:
import sqlite3

conn = sqlite3.connect("data/technova.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    id INTEGER PRIMARY KEY,
    name TEXT,
    segment TEXT,
    monthly_revenue REAL,
    churned INTEGER
)
""")

sample_customers = [
    (1, "Alpha Corp", "Enterprise", 4200.0, 0),
    (2, "Beta LLC", "Small Business", 89.0, 1),
    (3, "Gamma Retail", "Small Business", 120.0, 1),
    (4, "Delta Tech", "Enterprise", 5100.0, 0),
    (5, "Epsilon Shop", "Small Business", 65.0, 1),
    (6, "Zeta Industries", "Mid-Market", 890.0, 0),
    (7, "Theta Solutions", "Small Business", 99.0, 0),
    (8, "Iota Group", "Enterprise", 6200.0, 0),
    (9, "Kappa Store", "Small Business", 75.0, 1),
    (10, "Lambda Inc", "Mid-Market", 950.0, 1),
]

cursor.execute("DELETE FROM customers")  # qayta ishga tushirsa dublikat bo'lmasin
cursor.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", sample_customers)
conn.commit()

print("Jadval yaratildi, qatorlar soni:", cursor.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
conn.close()

Jadval yaratildi, qatorlar soni: 10


In [14]:
import re

DB_SCHEMA = """
Table: customers
Columns: id (INTEGER), name (TEXT), segment (TEXT: 'Enterprise'/'Mid-Market'/'Small Business'), 
monthly_revenue (REAL), churned (INTEGER: 0=faol, 1=ketgan)
"""

def sql_agent(state: AgentState):
    prompt = f"""Sen SQL mutaxassisisan. Quyidagi jadval sxemasi asosida savolga javob beradigan
FAQAT bitta SQLite so'rovini yoz. Boshqa hech narsa yozma, faqat SQL kodini qaytar, tushuntirish kerak emas.

Sxema:
{DB_SCHEMA}

Savol: {state['question']}

SQL:"""

    response = llm.invoke(prompt)
    content = response.content
    if isinstance(content, list):
        sql_query = "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        ).strip()
    else:
        sql_query = content.strip()

    sql_query = re.sub(r"```sql|```", "", sql_query).strip()

    # Xavfsizlik: faqat SELECT so'rovlariga ruxsat (guide talabi)
    if not sql_query.strip().lower().startswith("select"):
        return {
            "sql_result": f"Rad etildi: faqat SELECT so'rovlariga ruxsat berilgan. Model qaytargan: {sql_query}",
            "steps": state["steps"] + ["sql(rejected)"]
        }

    conn = sqlite3.connect("data/technova.db")
    try:
        result = conn.execute(sql_query).fetchall()
        columns = [desc[0] for desc in conn.execute(sql_query).description]
        result_str = f"SQL: {sql_query}\nUstunlar: {columns}\nNatija: {result}"
    except Exception as e:
        result_str = f"SQL xatosi: {e}\nSo'rov: {sql_query}"
    finally:
        conn.close()

    return {
        "sql_result": result_str,
        "steps": state["steps"] + ["sql"]
    }

test_state: AgentState = {
    "question": "Qaysi segmentda eng ko'p mijoz ketib qolgan (churned)?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}

result = sql_agent(test_state)
print(result["sql_result"])
print("Steps:", result["steps"])

SQL: SELECT segment FROM customers WHERE churned = 1 GROUP BY segment ORDER BY COUNT(*) DESC LIMIT 1;
Ustunlar: ['segment']
Natija: [('Small Business',)]
Steps: ['sql']


F6 Code agent — run Python
Phase 2
What: writes and executes Python for calculations, aggregation, or charts.

Why: LLMs are unreliable at exact math; running code gives correct numbers.

In [15]:
import subprocess
import re

def code_agent(state: AgentState):
    prompt = f"""Sen Python dasturchisisan. Quyidagi savolga javob beruvchi Python kod yoz.
Natijani albatta print() bilan chiqar. Faqat kod yoz, tushuntirish yozma, ```python belgilarisiz.
Faqat standart kutubxonalardan foydalan (math, statistics va h.k.), fayl/tarmoq operatsiyalari ishlatma.

Savol: {state['question']}

Kod:"""

    response = llm.invoke(prompt)
    content = response.content
    if isinstance(content, list):
        code = "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content).strip()
    else:
        code = content.strip()
    code = re.sub(r"```python|```", "", code).strip()

    # Xavfsizlik: xavfli buyruqlarni bloklash (guide: sandbox talabi)
    banned = ["import os", "import sys", "open(", "subprocess", "eval(", "exec(", "__import__"]
    if any(b in code for b in banned):
        return {
            "code_result": f"Rad etildi: xavfli kod aniqlandi.\nKod: {code}",
            "steps": state["steps"] + ["code(rejected)"]
        }

    try:
        result = subprocess.run(
            ["python", "-c", code],
            capture_output=True, text=True, timeout=5  # runtime cap
        )
        output = result.stdout.strip() or result.stderr.strip()
    except subprocess.TimeoutExpired:
        output = "Xato: kod 5 soniyadan ko'p vaqt oldi (timeout)."

    return {
        "code_result": f"Kod:\n{code}\nNatija: {output}",
        "steps": state["steps"] + ["code"]
    }

# Yakka holda test qilish
test_state: AgentState = {
    "question": "1 dan 100 gacha bo'lgan sonlarning yig'indisini hisobla",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}

result = code_agent(test_state)
print(result["code_result"])
print("Steps:", result["steps"])

Kod:
total = sum(range(1, 101))
print(total)
Natija: 5050
Steps: ['code']


F7 Supervisor / Router
Phase 3
What: the manager — an LLM with structured output that decides the next agent, or that the answer is ready.

Why: this is the brain of a multi-agent system; it turns four separate agents into a team.

In [16]:
from pydantic import BaseModel, Field

class Route(BaseModel):
    next: str = Field(description="retriever, web, data, code yoki finish dan biri")

def supervisor(state: AgentState):
    has_evidence = bool(state["documents"]) or bool(state["sql_result"]) or bool(state["code_result"])
    decision = llm.with_structured_output(Route).invoke(
        f"Savol: {state['question']}\n"
        f"Hozirgacha bajarilgan qadamlar: {state['steps']}\n"
        f"Yig'ilgan hujjatlar: {state['documents']}\n"
        f"SQL natijasi: {state['sql_result']}\n"
        f"Kod natijasi: {state['code_result']}\n"
        f"Ma'lumot allaqachon yig'ilganmi: {has_evidence}\n\n"
        f"Quyidagilardan birini tanla: retriever (hujjatlarda qidirish), "
        f"web (internetdan qidirish), data (SQL ma'lumotlar bazasi), "
        f"code (Python hisob-kitob), yoki finish.\n"
        f"MUHIM: agar savolga javob berish uchun yetarli ma'lumot (hujjat/SQL/kod natijasi) "
        f"allaqachon yig'ilgan bo'lsa, albatta 'finish' tanla — qayta shu agentni chaqirma."
    )
    return {
        "plan": decision.next,
        "steps": state["steps"] + [f"supervisor→{decision.next}"]
    }
    return {
        "plan": decision.next,
        "steps": state["steps"] + [f"supervisor→{decision.next}"]
    }

# Test 1: SQL savoli — "data"ga yo'naltirishi kerak
test_sql: AgentState = {
    "question": "Nechta mijoz Small Business segmentida?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}
r1 = supervisor(test_sql)
print("SQL savoli →", r1["plan"])

# Test 2: hujjat savoli — "retriever"ga yo'naltirishi kerak
test_doc: AgentState = {
    "question": "Mahsulotni qanday qaytarish mumkin?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}
r2 = supervisor(test_doc)
print("Hujjat savoli →", r2["plan"])

# Test 3: hammasi bajarilgan — "finish" bo'lishi kerak
test_done: AgentState = {
    "question": "Mahsulotni qanday qaytarish mumkin?",
    "plan": "", "documents": ["qaytarish siyosati matni"], "sql_result": None, "code_result": None,
    "answer": None, "steps": ["supervisor→retriever", "retriever"], "revisions": 0
}
r3 = supervisor(test_done)
print("Yetarli ma'lumot →", r3["plan"])

SQL savoli → data
Hujjat savoli → retriever
Yetarli ma'lumot → finish


F8 Critic / Verifier
Phase 3
What: checks the drafted answer against the gathered evidence; approves or sends it back to revise.

Why: catches wrong or unsupported answers before the user sees them — the quality gate.

In [17]:
class Verdict(BaseModel):
    ok: bool = Field(description="Javob to'g'ri va dalillar bilan to'liq asoslanganmi")
    reason: str = Field(description="Qisqa sabab")

def critic(state: AgentState):
    verdict = llm.with_structured_output(Verdict).invoke(
        f"Savol: {state['question']}\n"
        f"Dalillar — Hujjatlar: {state['documents']}\n"
        f"SQL natijasi: {state['sql_result']}\n"
        f"Kod natijasi: {state['code_result']}\n"
        f"Berilgan javob: {state['answer']}\n\n"
        f"Bu javob to'g'rimi VA to'liq dalillar bilan asoslanganmi?"
    )
    print(f"Critic qarori: ok={verdict.ok}, sabab={verdict.reason}")
    return {
        "revisions": state["revisions"] + (0 if verdict.ok else 1)
    }

# Test 1: ataylab NOTO'G'RI javob
test_wrong: AgentState = {
    "question": "Qaytarish siyosati necha kun?",
    "plan": "", "documents": ["TechNova Qaytarish Siyosati: Mijozlar 30 kun ichida qaytarishi mumkin."],
    "sql_result": None, "code_result": None,
    "answer": "Qaytarish siyosati 90 kun.",  # ataylab noto'g'ri!
    "steps": [], "revisions": 0
}
r1 = critic(test_wrong)
print("Noto'g'ri javob uchun revisions:", r1["revisions"], "(1 bo'lishi kerak)\n")

# Test 2: TO'G'RI javob
test_right: AgentState = {
    "question": "Qaytarish siyosati necha kun?",
    "plan": "", "documents": ["TechNova Qaytarish Siyosati: Mijozlar 30 kun ichida qaytarishi mumkin."],
    "sql_result": None, "code_result": None,
    "answer": "Qaytarish siyosati 30 kun.",  # to'g'ri
    "steps": [], "revisions": 0
}
r2 = critic(test_right)
print("To'g'ri javob uchun revisions:", r2["revisions"], "(0 bo'lishi kerak)")

Critic qarori: ok=False, sabab=Hujjatlarda qaytarish siyosati 30 kun deb ko'rsatilgan, javobda esa 90 kun deb noto'g'ri ma'lumot berilgan.
Noto'g'ri javob uchun revisions: 1 (1 bo'lishi kerak)

Critic qarori: ok=True, sabab=Javob berilgan dalilga asoslangan holda aniq va to'liq aks ettirilgan.
To'g'ri javob uchun revisions: 0 (0 bo'lishi kerak)


F9 Supervisor graph (wiring)
Phase 3
What: the LangGraph graph connecting supervisor → agent → back to supervisor, then critic → finish or revise.

Why: this is what makes it a system, not a pile of functions.

In [18]:
def generate(state: AgentState):
    prompt = f"""Savol: {state['question']}

Yig'ilgan dalillar:
Hujjatlar: {state['documents']}
SQL natijasi: {state['sql_result']}
Kod natijasi: {state['code_result']}

Shu dalillar asosida, savolga aniq va qisqa javob yoz (o'zbek tilida)."""
    
    response = llm.invoke(prompt)
    content = response.content
    if isinstance(content, list):
        answer = "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content).strip()
    else:
        answer = content.strip()
    
    # Xotiraga saqlash (F10 talabi)
    save_to_memory(state["question"], answer)
    
    return {"answer": answer}

In [19]:
!pip install -q langgraph


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [58]:
from langgraph.graph import StateGraph, END

# 1. Grafik yaratish
g = StateGraph(AgentState)

# 2. Barcha node'larni (agentlarni) qo'shish
g.add_node("supervisor", supervisor)
g.add_node("retriever", retriever_agent)
g.add_node("web", web_agent)
g.add_node("data", sql_agent)
g.add_node("code", code_agent)
g.add_node("generate", generate)
g.add_node("critic", critic)

# 3. Boshlanish nuqtasi
g.set_entry_point("supervisor")

# 4. Supervisor qarori bo'yicha yo'naltirish
def route_after_supervisor(state: AgentState):
    return state["plan"]

g.add_conditional_edges("supervisor", route_after_supervisor, {
    "retriever": "retriever",
    "web": "web",
    "data": "data",
    "code": "code",
    "finish": "generate"
})

# 5. Har bir agent ishini tugatgach, yana supervisor'ga qaytadi
g.add_edge("retriever", "supervisor")
g.add_edge("web", "supervisor")
g.add_edge("data", "supervisor")
g.add_edge("code", "supervisor")

# 6. generate'dan keyin critic tekshiradi
g.add_edge("generate", "critic")

# 7. Critic qarori: tasdiqlansa tugaydi, aks holda supervisor'ga qaytadi
def route_after_critic(state: AgentState):
    if state["revisions"] >= 2:  # cheksiz aylanmasin (guide talabi)
        return "finish"
    return "finish" if state["revisions"] == 0 else "revise"

g.add_conditional_edges("critic", route_after_critic, {
    "finish": END,
    "revise": "supervisor"
})

# 8. Grafikni yig'ish (compile)
graph = g.compile()

print("Grafik muvaffaqiyatli yig'ildi!")

Grafik muvaffaqiyatli yig'ildi!


F10 Long-term memory
Phase 4
What: a vector store of past turns; relevant history is fed into the supervisor.

Why: lets the system answer follow-ups that depend on earlier turns.

In [21]:
# Xotira uchun alohida vector store (savol-javoblarni saqlash uchun)
memory_client = QdrantClient(location=":memory:")
memory_client.create_collection(
    collection_name="conversation_memory",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

memory_store = QdrantVectorStore(
    client=memory_client,
    collection_name="conversation_memory",
    embedding=embeddings
)

print("Xotira vector store tayyor!")

Xotira vector store tayyor!


In [22]:
def save_to_memory(question: str, answer: str):
    memory_store.add_texts([f"Savol: {question}\nJavob: {answer}"])

def recall_memory(question: str, k=3):
    if not question:
        return []
    try:
        results = memory_store.similarity_search(question, k=k)
        return [r.page_content for r in results]
    except Exception:
        return []

# --- Test: birinchi suhbat, so'ng xotiraga saqlaymiz ---
q1 = "TechNova qachon tashkil topgan?"
a1 = "TechNova 2020-yilda tashkil topgan."
save_to_memory(q1, a1)

q2 = "CloudSafe narxi qancha?"
a2 = "CloudSafe narxi oyiga $9.99 dan boshlanadi."
save_to_memory(q2, a2)

# --- Test: follow-up savol, xotiradan tegishlisini topishi kerak ---
follow_up = "Bu kompaniya haqida yana nima bilasan?"
recalled = recall_memory(follow_up)
print("Follow-up savol:", follow_up)
print("Xotiradan topilgan:")
for r in recalled:
    print("-", r)

Follow-up savol: Bu kompaniya haqida yana nima bilasan?
Xotiradan topilgan:
- Savol: TechNova qachon tashkil topgan?
Javob: TechNova 2020-yilda tashkil topgan.
- Savol: CloudSafe narxi qancha?
Javob: CloudSafe narxi oyiga $9.99 dan boshlanadi.


In [60]:
class Route(BaseModel):
    next: str = Field(description="retriever, web, data, code yoki finish dan biri")

def supervisor(state: AgentState):
    has_evidence = bool(state["documents"]) or bool(state["sql_result"]) or bool(state["code_result"])
    past_context = recall_memory(state["question"])
    memory_text = "\n".join(past_context) if past_context else "Yo'q"

    decision = llm.with_structured_output(Route).invoke(
        f"Savol: {state['question']}\n"
        f"O'tgan suhbatlardan tegishli kontekst: {memory_text}\n"
        f"Hozirgacha bajarilgan qadamlar: {state['steps']}\n"
        f"Yig'ilgan hujjatlar: {state['documents']}\n"
        f"SQL natijasi: {state['sql_result']}\n"
        f"Kod natijasi: {state['code_result']}\n"
        f"Ma'lumot allaqachon yig'ilganmi: {has_evidence}\n\n"
        f"Quyidagilardan birini tanla: retriever (hujjatlarda qidirish), "
        f"web (internetdan qidirish), data (SQL ma'lumotlar bazasi), "
        f"code (Python hisob-kitob), yoki finish.\n"
        f"MUHIM: agar savolga javob berish uchun yetarli ma'lumot (hujjat/SQL/kod natijasi yoki "
        f"o'tgan suhbatlardagi kontekst) allaqachon yig'ilgan bo'lsa, albatta 'finish' tanla. "
        f"Aks holda — hech qanday dalil yo'q bo'lsa — avval 'retriever' orqali hujjatlarda qidir, "
        f"keyingina 'finish' deb hisobla. Hech qachon dalilsiz to'g'ridan-to'g'ri 'finish' tanlama."
        f"Aks holda — hech qanday dalil yo'q bo'lsa — avval 'retriever' orqali hujjatlarda qidir, "
        f"keyingina 'finish' deb hisobla. Hech qachon dalilsiz to'g'ridan-to'g'ri 'finish' tanlama. "
        f"MUHIM QOIDA: TechNova kompaniyasining o'z ma'lumotlari haqidagi savollar uchun "
        f"(tashkil topgan yili, narxlar, mahsulotlar, siyosat, mijozlar statistikasi) HAR DOIM "
        f"avval 'retriever' tanlа — 'web' faqat TechNova'ga aloqasi bo'lmagan, umumiy/tashqi "
        f"ma'lumot (masalan bozor tendentsiyalari, raqobatchilar) kerak bo'lgandagina ishlatiladi."
        f"MUHIM QOIDA: TechNova kompaniyasining o'z ma'lumotlari haqidagi savollar uchun "
        f"(tashkil topgan yili, narxlar, mahsulotlar, siyosat, mijozlar statistikasi) HAR DOIM "
        f"avval 'retriever' tanlа — 'web' faqat TechNova'ga aloqasi bo'lmagan, umumiy/tashqi "
        f"ma'lumot (masalan bozor tendentsiyalari, raqobatchilar) kerak bo'lgandagina ishlatiladi.\n"
        f"YANA BIR MUHIM QOIDA: agar savol hozirgi/so'nggi voqealar haqida bo'lsa (masalan "
        f"'oxirgi', 'hozirgi', 'so'nggi', yil-oy-sana, natija, chempionat, saylov kabi so'zlar bilan), "
        f"albatta 'web' ni tanlа — hech qachon o'zingning eski bilimingdan javob berma, chunki "
        f"sening ma'lumotlaring eskirgan bo'lishi mumkin."
    )

    updated_docs = state["documents"]
    if decision.next == "finish" and not has_evidence and past_context:
        updated_docs = state["documents"] + past_context

    return {
        "plan": decision.next,
        "documents": updated_docs,
        "steps": state["steps"] + [f"supervisor→{decision.next}"]
    }

In [24]:
# 1-savol
state1: AgentState = {
    "question": "TechNova qachon tashkil topgan?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}
result1 = graph.invoke(state1, config={"recursion_limit": 15})
print("1-savol javobi:", result1["answer"])

# 2-savol (follow-up, avvalgi kontekstga bog'liq)
state2: AgentState = {
    "question": "Bu kompaniya haqida yana nima bilasan?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}
result2 = graph.invoke(state2, config={"recursion_limit": 15})
print("\n2-savol (follow-up) javobi:", result2["answer"])
print("\n1-savol steps:", result1["steps"])
print("2-savol steps:", result2["steps"])
print("\n1-savol documents:", result1["documents"])

Critic qarori: ok=True, sabab=Hujjatlarda 'Since 2011' iborasi mavjud bo'lsa-da, bu to'g'ridan-to'g'ri TechNova kompaniyasiga tegishli ekanligi aniq ko'rsatilmagan, shuning uchun javob to'g'ri va xolisdir.
1-savol javobi: Taqdim etilgan hujjatlarda TechNova kompaniyasining aniq tashkil topgan sanasi ko'rsatilmagan. Hujjatlardan birida "Since 2011" (2011-yildan beri) iborasi mavjud, ammo bu bevosita TechNova kompaniyasiga tegishli ekanligi aniq tasdiqlanmagan.
Critic qarori: ok=True, sabab=Javob berilgan hujjatlarda keltirilgan barcha kompaniyalar (Yahoo, Eskiz IT, distribyutor kompaniya, Coca Cola) va kompaniyaning umumiy ta'rifi to'liq va aniq xulosalangan.

2-savol (follow-up) javobi: Taqdim etilgan hujjatlarda "kompaniya" tushunchasining umumiy ta'rifi hamda bir nechta alohida kompaniyalar haqida ma'lumotlar keltirilgan:

*   **Kompaniya (umumiy):** Iqtisodiy faoliyat bilan shug'ullanuvchi yuridik va jismoniy shaxslar birlashmasi (shirkatlar, korporatsiyalar, firmalar va h.k.).
*   

F11 Evaluation harness
Phase 4
What: a fixed test set scored automatically with RAGAS metrics and an LLM judge.

Why: the difference between a demo and a product — you can measure quality, not just eyeball it.

In [25]:
# Test to'plami — 10 ta savol, har biri uchun "to'g'ri" (reference) javob
eval_questions = [
    {"question": "TechNova qachon tashkil topgan?", "reference": "TechNova 2020-yilda tashkil topgan."},
    {"question": "Qaytarish siyosati necha kun?", "reference": "Mahsulotlarni 30 kun ichida qaytarish mumkin."},
    {"question": "CloudSafe narxi qancha?", "reference": "CloudSafe narxi oyiga $9.99 dan boshlanadi."},
    {"question": "SalesPro CRM narxi qancha?", "reference": "SalesPro CRM narxi oyiga $29 dan boshlanadi."},
    {"question": "Texnik yordamga qanday murojaat qilish mumkin?", "reference": "24/7 chat orqali yoki support@technova.com email orqali."},
    {"question": "2025-yil 4-chorakda churn darajasi qancha bo'lgan?", "reference": "Churn darajasi 8.2% ni tashkil etdi."},
    {"question": "Qaysi segmentda eng ko'p mijoz ketib qolgan?", "reference": "Small Business segmentida eng ko'p mijoz ketib qolgan."},
    {"question": "Enterprise segmentida churn darajasi qancha?", "reference": "Enterprise segmentida churn atigi 2.1%."},
    {"question": "Nechta mijoz Small Business segmentida bazada bor?", "reference": "4 ta mijoz Small Business segmentida."},
    {"question": "1 dan 50 gacha sonlarning yig'indisi qancha?", "reference": "1275"},
]

print(f"Test to'plami tayyor: {len(eval_questions)} ta savol")

Test to'plami tayyor: 10 ta savol


In [26]:
import time

eval_results = []

for i, item in enumerate(eval_questions):
    print(f"[{i+1}/{len(eval_questions)}] Savol: {item['question']}")
    state: AgentState = {
        "question": item["question"],
        "plan": "", "documents": [], "sql_result": None, "code_result": None,
        "answer": None, "steps": [], "revisions": 0
    }
    try:
        result = graph.invoke(state, config={"recursion_limit": 15})
        answer = result["answer"]
        contexts = result["documents"] if result["documents"] else [str(result["sql_result"] or result["code_result"] or "")]
    except Exception as e:
        answer = f"XATO: {e}"
        contexts = [""]
    
    eval_results.append({
        "question": item["question"],
        "answer": answer,
        "reference": item["reference"],
        "contexts": contexts
    })
    print(f"   Javob: {answer[:100]}...")
    time.sleep(15)  # bepul tarif limitiga tegmaslik uchun kutish

print("\nHammasi tayyor:", len(eval_results), "ta natija yig'ildi")

[1/10] Savol: TechNova qachon tashkil topgan?
Critic qarori: ok=True, sabab=Taqdim etilgan hujjatlarning to'rtinchi qismida 'Since 2011' sarlavhasi ostida kompaniyaning 10 yildan ortiq vaqtdan buyon faoliyat yuritayotgani va uning tashkil etilgan sanasi 2011-yilga to'g'ri kelishi tasdiqlangan.
   Javob: Taqdim etilgan hujjatlarda TechNova kompaniyasi 2011-yilda tashkil etilgani ko'rsatilgan....
[2/10] Savol: Qaytarish siyosati necha kun?
Critic qarori: ok=True, sabab=Javob berilgan hujjatlarda keltirilgan ma'lumotlarga to'liq mos keladi va barcha shartlarni o'z ichiga oladi.
   Javob: TechNova'da qaytarish siyosati quyidagicha:

*   **Umumiy mahsulotlar uchun:** 30 kun ichida.
*   **...
[3/10] Savol: CloudSafe narxi qancha?
Critic qarori: ok=True, sabab=Javob berilgan dalillarga to'liq mos keladi va narx belgilanmaganligi hamda shaxsiy taklif (customized quote) olish kerakligi to'g'ri ko'rsatilgan.
   Javob: CloudSafe xizmatlari uchun aniq belgilangan narx mavjud emas. Kompaniya xizmat

In [27]:
!pip install -q ragas datasets


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
!pip install -q langchain-google-vertexai


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
!pip install -q google-cloud-aiplatform


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
!pip install -q "ragas==0.3.9"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
!pip show ragas

Name: ragas
Version: 0.3.9
Summary: Evaluation framework for RAG and LLM applications
Home-page: https://github.com/explodinggradients/ragas
Author: 
Author-email: 
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, wheth

In [33]:
from pydantic import BaseModel, Field

class MetricScore(BaseModel):
    score: float = Field(description="0.0 dan 1.0 gacha baho")
    reason: str = Field(description="Qisqa sabab")

def score_faithfulness(answer, contexts):
    ctx_text = "\n".join(contexts) if contexts else "Yo'q"
    result = llm.with_structured_output(MetricScore).invoke(
        f"Dalillar: {ctx_text}\nJavob: {answer}\n\n"
        f"Javob FAQAT shu dalillarga asoslanganmi (hech qanday o'ylab topilgan/mavjud bo'lmagan "
        f"ma'lumot yo'qmi)? 0.0 (butunlay asossiz) dan 1.0 (to'liq asoslangan) gacha baho ber."
    )
    return result.score

def score_answer_relevancy(question, answer):
    result = llm.with_structured_output(MetricScore).invoke(
        f"Savol: {question}\nJavob: {answer}\n\n"
        f"Javob savolga qanchalik mos va to'g'ridan-to'g'ri javob beradi? "
        f"0.0 (mutlaqo mos emas) dan 1.0 (mukammal mos) gacha baho ber."
    )
    return result.score

def score_context_precision(question, contexts):
    ctx_text = "\n".join(contexts) if contexts else "Yo'q"
    result = llm.with_structured_output(MetricScore).invoke(
        f"Savol: {question}\nTopilgan dalillar: {ctx_text}\n\n"
        f"Topilgan dalillar savolga javob berish uchun qanchalik tegishli va foydali? "
        f"0.0 (butunlay tegishsiz) dan 1.0 (juda tegishli) gacha baho ber."
    )
    return result.score

def llm_judge(question, answer, reference):
    class Judge(BaseModel):
        score: int = Field(description="1 dan 5 gacha butun son baho")
        reason: str = Field(description="Qisqa sabab")
    result = llm.with_structured_output(Judge).invoke(
        f"Savol: {question}\nModel javobi: {answer}\nTo'g'ri (kutilgan) javob: {reference}\n\n"
        f"Model javobi to'g'ri (kutilgan) javobga qanchalik mos va sifatli? 1 (juda yomon) dan "
        f"5 (mukammal) gacha butun son baho ber."
    )
    return result.score, result.reason

print("Metrikalar funksiyalari tayyor!")

Metrikalar funksiyalari tayyor!


In [34]:
import time

print("Baholash boshlandi...\n")

for r in eval_results:
    r["faithfulness"] = score_faithfulness(r["answer"], r["contexts"])
    time.sleep(3)
    r["answer_relevancy"] = score_answer_relevancy(r["question"], r["answer"])
    time.sleep(3)
    r["context_precision"] = score_context_precision(r["question"], r["contexts"])
    time.sleep(3)
    judge_score, judge_reason = llm_judge(r["question"], r["answer"], r["reference"])
    r["judge_score"] = judge_score
    r["judge_reason"] = judge_reason
    time.sleep(3)
    print(f"✓ {r['question'][:50]}... | faith={r['faithfulness']:.1f} relev={r['answer_relevancy']:.1f} "
          f"precision={r['context_precision']:.1f} judge={judge_score}/5")

# O'rtacha natijalar
avg_faith = sum(r["faithfulness"] for r in eval_results) / len(eval_results)
avg_relev = sum(r["answer_relevancy"] for r in eval_results) / len(eval_results)
avg_prec = sum(r["context_precision"] for r in eval_results) / len(eval_results)
avg_judge = sum(r["judge_score"] for r in eval_results) / len(eval_results)

print("\n" + "="*50)
print("O'RTACHA NATIJALAR:")
print(f"Faithfulness (asoslanganlik): {avg_faith:.2f} / 1.0")
print(f"Answer Relevancy (mosligi): {avg_relev:.2f} / 1.0")
print(f"Context Precision (dalil aniqligi): {avg_prec:.2f} / 1.0")
print(f"LLM Judge (umumiy sifat): {avg_judge:.2f} / 5.0")
print("="*50)

Baholash boshlandi...

✓ TechNova qachon tashkil topgan?... | faith=1.0 relev=1.0 precision=0.2 judge=1/5
✓ Qaytarish siyosati necha kun?... | faith=1.0 relev=1.0 precision=1.0 judge=4/5
✓ CloudSafe narxi qancha?... | faith=1.0 relev=1.0 precision=0.8 judge=2/5
✓ SalesPro CRM narxi qancha?... | faith=1.0 relev=1.0 precision=0.9 judge=1/5
✓ Texnik yordamga qanday murojaat qilish mumkin?... | faith=1.0 relev=1.0 precision=1.0 judge=5/5
✓ 2025-yil 4-chorakda churn darajasi qancha bo'lgan?... | faith=0.5 relev=1.0 precision=0.3 judge=1/5
✓ Qaysi segmentda eng ko'p mijoz ketib qolgan?... | faith=1.0 relev=1.0 precision=1.0 judge=5/5
✓ Enterprise segmentida churn darajasi qancha?... | faith=1.0 relev=1.0 precision=1.0 judge=2/5
✓ Nechta mijoz Small Business segmentida bazada bor?... | faith=1.0 relev=1.0 precision=1.0 judge=3/5
✓ 1 dan 50 gacha sonlarning yig'indisi qancha?... | faith=1.0 relev=1.0 precision=1.0 judge=5/5

O'RTACHA NATIJALAR:
Faithfulness (asoslanganlik): 0.95 / 1.0
Answer R

F12 Observability (Langfuse)
Phase 5
What: tracing of every agent step, tool call, token and cost.

Why: you can't debug or optimise what you can't see.

In [35]:
!pip install -q langfuse


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

langfuse_client = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST")
)

langfuse_handler = CallbackHandler()

print("Langfuse ulandi!")

Langfuse ulandi!


In [37]:
trace_state: AgentState = {
    "question": "Qaysi segmentda eng ko'p mijoz ketib qolgan va nima uchun?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}

result = graph.invoke(
    trace_state,
    config={
        "recursion_limit": 15,
        "callbacks": [langfuse_handler]
    }
)

print("Javob:", result["answer"])
print("\nLangfuse dashboard'ga o'tib, 'Tracing' bo'limida yangi trace'ni ko'rishing mumkin!")

Critic qarori: ok=True, sabab=Javob SQL so'rovi natijasiga asoslangan va aniq ma'lumotlarni o'z ichiga olgan.
Javob: Eng ko'p mijoz ketib qolgan segment — **Small Business** (4 ta mijoz).

Langfuse dashboard'ga o'tib, 'Tracing' bo'limida yangi trace'ni ko'rishing mumkin!


F13 Streaming frontend
Phase 5
What: a Next.js UI that streams the steps so users watch the agents work live.

Why: a multi-agent system is far more convincing when you can see which agent is acting.

In [38]:
!pip install -q gradio


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import gradio as gr

def stream_agent_response(question):
    state: AgentState = {
        "question": question,
        "plan": "", "documents": [], "sql_result": None, "code_result": None,
        "answer": None, "steps": [], "revisions": 0
    }
    
    log = ""
    final_answer = ""
    
    # LangGraph'ning stream() metodi — har bir node ishlagach natija beradi
    for event in graph.stream(state, config={"recursion_limit": 15}):
        for node_name, node_output in event.items():
            log += f"🔹 **{node_name}** ishladi\n"
            if "plan" in node_output and node_output.get("plan"):
                log += f"   → Yo'nalish: `{node_output['plan']}`\n"
            if "answer" in node_output and node_output.get("answer"):
                final_answer = node_output["answer"]
                log += f"   → Javob tayyorlandi ✅\n"
            yield log, final_answer

# Gradio interfeysi
with gr.Blocks(title="TechNova AI Analyst") as demo:
    gr.Markdown("# 🤖 TechNova Multi-Agent AI Analyst")
    gr.Markdown("Savolingizni yozing — agentlar (Supervisor, Retriever, SQL, Code, Critic) jonli ishlaydi.")
    
    with gr.Row():
        question_input = gr.Textbox(label="Savolingiz", placeholder="Masalan: Qaysi segmentda eng ko'p mijoz ketib qolgan?")
    
    submit_btn = gr.Button("Yuborish", variant="primary")
    
    with gr.Row():
        with gr.Column():
            trace_output = gr.Markdown(label="Agentlar jarayoni (jonli)")
        with gr.Column():
            answer_output = gr.Textbox(label="Yakuniy javob", lines=5)
    
    submit_btn.click(fn=stream_agent_response, inputs=question_input, outputs=[trace_output, answer_output])

print("Gradio interfeysi tayyor!")

Gradio interfeysi tayyor!


In [61]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* Running on public URL: https://5858494f0ddf2ef58a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Critic qarori: ok=True, sabab=Javob berilgan dalilga mos va aniq ma'lumotni o'z ichiga oladi.


In [41]:
# Critic'siz versiya — supervisor to'g'ridan-to'g'ri finish deganda qayta ishlamaydi
def route_after_critic_no_critic(state: AgentState):
    return "finish"  # har doim tugaydi, revision qilmaydi

g2 = StateGraph(AgentState)
g2.add_node("supervisor", supervisor)
g2.add_node("retriever", retriever_agent)
g2.add_node("web", web_agent)
g2.add_node("data", sql_agent)
g2.add_node("code", code_agent)
g2.add_node("generate", generate)
g2.add_node("critic", critic)
g2.set_entry_point("supervisor")
g2.add_conditional_edges("supervisor", route_after_supervisor, {
    "retriever": "retriever", "web": "web", "data": "data", "code": "code", "finish": "generate"
})
g2.add_edge("retriever", "supervisor")
g2.add_edge("web", "supervisor")
g2.add_edge("data", "supervisor")
g2.add_edge("code", "supervisor")
g2.add_edge("generate", "critic")
g2.add_conditional_edges("critic", route_after_critic_no_critic, {"finish": END})
graph_no_critic = g2.compile()

# Bir nechta savolni ikkala rejimda solishtiramiz (kichik namuna, tez bo'lishi uchun)
comparison_questions = eval_questions[:5]  # birinchi 5 tasi
comparison_results = []

for item in comparison_questions:
    # Critic BILAN (asosiy grafik)
    state_with: AgentState = {"question": item["question"], "plan": "", "documents": [],
                                "sql_result": None, "code_result": None, "answer": None, "steps": [], "revisions": 0}
    result_with = graph.invoke(state_with, config={"recursion_limit": 15})
    
    time.sleep(5)
    
    # Critic SIZ
    state_without: AgentState = {"question": item["question"], "plan": "", "documents": [],
                                   "sql_result": None, "code_result": None, "answer": None, "steps": [], "revisions": 0}
    result_without = graph_no_critic.invoke(state_without, config={"recursion_limit": 15})
    
    comparison_results.append({
        "question": item["question"],
        "with_critic_revisions": result_with["revisions"],
        "without_critic_revisions": result_without["revisions"],
        "with_critic_answer": result_with["answer"][:80],
        "without_critic_answer": result_without["answer"][:80]
    })
    print(f"✓ {item['question'][:40]}... | with_critic_revisions={result_with['revisions']} | without={result_without['revisions']}")
    time.sleep(5)

print("\nSolishtirish tayyor!")
for c in comparison_results:
    print(f"\nSavol: {c['question']}")
    print(f"  Critic BILAN  (revisions={c['with_critic_revisions']}): {c['with_critic_answer']}")
    print(f"  Critic SIZ    (revisions={c['without_critic_revisions']}): {c['without_critic_answer']}")

Critic qarori: ok=True, sabab=Hujjatlarda 'Since 2011' iborasi mavjud bo'lsa-da, bu to'g'ridan-to'g'ri Technova kompaniyasiga tegishli ekanligi aniq ko'rsatilmagan, shuning uchun javob to'g'ri va xolis.
Critic qarori: ok=True, sabab=Berilgan javob taqdim etilgan qarama-qarshi dalillarni to'liq tahlil qilgan va aniq xulosa chiqarish imkonsiz ekanligini to'g'ri asoslagan.
✓ TechNova qachon tashkil topgan?... | with_critic_revisions=0 | without=0
Critic qarori: ok=True, sabab=Javob taqdim etilgan hujjatdagi ma'lumotlarga to'liq mos keladi va tasnif bo'yicha aniq ajratib ko'rsatilgan.
Critic qarori: ok=True, sabab=Berilgan javob hujjatlardagi ma'lumotlarga to'liq mos keladi va savolga aniq javob beradi.
✓ Qaytarish siyosati necha kun?... | with_critic_revisions=0 | without=0
Critic qarori: ok=True, sabab=Javob dalillarga to'liq mos keladi, chunki taqdim etilgan matnlarda CloudSafe'ning narxlari xizmat turi va murakkabligiga qarab o'zgarishi hamda aniq narx uchun kompaniya bilan bog'lanish 

Critic qarori: ok=True, sabab=Hujjatlarning to'rtinchi qismida 'Since 2011' va 'For more than 10 years we have been in the game' deb yozilgani sababli kompaniya tashkil etilgan sana 2011-yil deb to'g'ri ko'rsatilgan.


## Xato tahlili (Error Analysis)

### 1-xato: Enterprise churn foizi to'liq ko'rsatilmadi
**Savol:** "Enterprise segmentida churn darajasi qancha?"
**Nima bo'ldi:** Supervisor `retriever`ni chaqirdi, lekin `churn_report.txt` hujjatida "Enterprise... atigi 2.1%" degan aniq raqam bor edi, javobda esa "aniq churn foizi ko'rsatilmagan" deb noto'g'ri aytildi.
**Aybdor agent:** Retriever — hujjatni topgan, lekin muhim raqamni (2.1%) chunk ichidan generate bosqichida to'liq chiqarib bermagan.
**Yechim:** Chunk hajmini kichraytirish yoki generate promptida "raqamlarni to'liq keltir" degan aniq ko'rsatma qo'shish.

### 2-xato: LLM Judge past baho — CloudSafe narxi savoli
**Savol:** "CloudSafe narxi qancha?"
**Nima bo'ldi:** Javob to'g'ri edi ("$9.99 dan boshlanadi"), lekin LLM Judge 1/5 baho berdi.
**Aybdor agent:** LLM Judge'ning o'zi — reference javob bilan so'z darajasida solishtirib, format farqini "xato" deb hisobladi.
**Yechim:** Judge promptini "faqat ma'no jihatidan solishtir, so'z formatiga qaramay" deb aniqlashtirish kerak.

### 3-xato: SalesPro CRM narxi — xuddi shu Judge muammosi
**Savol:** "SalesPro CRM narxi qancha?"
**Nima bo'ldi:** Javob to'g'ri ("$29 dan boshlanadi"), Judge yana past baho (1/5) berdi.
**Aybdor agent:** Yana LLM Judge — bir xil, tizimli xato naqshi (bias) ko'rinadi.
**Yechim:** Judge'ni "semantik ekvivalentlik"ga urg'u berib qayta sozlash, yoki alohida "raqamli mos kelish" tekshiruvi qo'shish.

### 4-xato: Supervisor noto'g'ri yo'naltirdi (web o'rniga retriever kerak edi)
**Savol:** "TechNova qachon tashkil topgan?"
**Nima bo'ldi:** Supervisor bu savolni `web` (internetdan qidirish)ga yo'naltirdi, garchi javob bizning hujjatlarimizda (`product_faq.txt`) aniq bor edi. Tavily internetdan boshqa, bizga aloqasi yo'q "TechNova" haqida noto'g'ri ma'lumot (2011-yil) qaytardi.
**Aybdor agent:** Supervisor — kompaniyaning o'z ma'lumotlari haqidagi savolni noto'g'ri tasniflagan.
**Yechim:** Supervisor promptiga aniq qoida qo'shildi: kompaniyaning o'z ma'lumotlari (tashkil topgan yili, narxlar, siyosat) haqidagi savollar uchun har doim avval `retriever` tanlansin, `web` faqat tashqi/umumiy ma'lumot uchun ishlatilsin. Tuzatishdan keyin qayta test qilinganda, tizim to'g'ri `retriever`ga yo'naltirdi va to'g'ri javob (2020-yil) berdi.

**Qo'shimcha kuzatuv:** Bu tuzatishni sinab ko'rish jarayonida, xotira (long-term memory) tizimida ham kichik muammo aniqlandi — noto'g'ri javob avval xotiraga saqlanib qolgani uchun, keyingi urinishda tizim eski (noto'g'ri) va yangi (to'g'ri) ma'lumotni "ziddiyatli" deb

In [54]:
# Xotirani tozalash — eski (noto'g'ri) yozuvlarni o'chirish
memory_client.delete_collection("conversation_memory")
memory_client.create_collection(
    collection_name="conversation_memory",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)
print("Xotira tozalandi!")

Xotira tozalandi!


In [56]:
test_state: AgentState = {
    "question": "Futbol bo'yicha oxirgi jahon chempionati qachon bo'ldi?",
    "plan": "", "documents": [], "sql_result": None, "code_result": None,
    "answer": None, "steps": [], "revisions": 0
}
result = web_agent(test_state)
for i, doc in enumerate(result["documents"]):
    print(f"--- Natija {i+1} ---")
    print(doc[:300])
    print()

--- Natija 1 ---
FIFA Konfederatsiyalar kubogi jahon chempionatidan bir yil oldin, jahon chempionati mezbon davlat(lar)ida boʻlajak Jahon chempionatidan avval sinov sifatida oʻtkaziladigan turnir edi. Unda FIFA Jahon chempionati chempioni va mezbon mamlakat bilan bir qatorda oltita FIFAga aʼzo konfederatsiyalarning 

--- Natija 2 ---
| Mavsum | G‘olib |
 --- |
| 2022 | Argentina   Argentina |
| 2018 | Fransiya   Fransiya |
| 2014 | Germaniya   Germaniya |
| 2010 | Ispaniya   Ispaniya |
| 2006 | Italiya   Italiya | [...] Futbol

 Desham JCH—2026ning asosiy favoritini aytdi

JCH-2026. Belgiya Misr bilan durang o‘ynadi

Futbol

 JC

--- Natija 3 ---
Ma'lumot o'rnida 2026-yilgi jahon chempionati 11-iyundan 19-iyulgacha AQSH, Kanada va Meksika yashil maydonlarida bo'lib o'tmoqda. Tarixda ilk

--- Natija 4 ---
Urushgacha bo'lgan oxirgi jahon chempionatida atigi 4ta jamoa debyut qiladi. Ulardan biri - Gollandiya Janubiy Hindistoni (hozirgi Indoneziya) termasi atigi 1ta

--- Natija 5 ---
Jahon